# 🎨 Image Generation Explorer

**Run on Google Colab → Runtime → Change runtime type → T4 GPU** (free tier is fine)

| Section | Concept | What you'll see |
|---------|---------|----------------|
| **1** | Stable Diffusion — Denoising | Step-by-step image emergence from pure noise |
| **2** | VAE Latent Space | Encoding images, visualising & interpolating in latent space |
| **3** | DALL·E via OpenAI API | Text → image with `gpt-image-1`, side-by-side prompt comparison |
| **4** | Semantic Prompt Similarity | Sentence-Transformers embeddings + cosine similarity heatmap |

> **Section 3 only:** Add your OpenAI API key via **Colab Secrets** (🔑 icon in the left sidebar) with the name `OPENAI_API_KEY`.

In [ ]:
# ── Install dependencies (only needed in Colab) ──────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q diffusers==0.36.0 transformers accelerate sentence-transformers openai seaborn

import os, warnings, base64
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
warnings.filterwarnings('ignore')
print('✅ Ready.')

---
## Section 1 — Stable Diffusion: Watching Denoising Happen

Stable Diffusion starts from **pure Gaussian noise** and removes noise step-by-step, guided by a text prompt.  
We hook into the pipeline with a **callback** to decode the latent at selected timesteps — revealing the image as it emerges from chaos.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32
print(f'Device: {device}  |  dtype: {dtype}')

# CompVis/stable-diffusion-v1-4 is public and requires no HF token
pipe = StableDiffusionPipeline.from_pretrained(
    'CompVis/stable-diffusion-v1-4',
    torch_dtype=dtype,
    safety_checker=None,
).to(device)
pipe.set_progress_bar_config(disable=True)
print('✅ Pipeline loaded.')

In [ ]:
PROMPT     = 'a glowing astronaut on a violet alien planet, cinematic lighting'
NEG_PROMPT = 'blurry, low quality, watermark'
NUM_STEPS  = 30
SNAP_STEPS = [1, 4, 8, 15, 20, 25, 29]
SEED       = 42

snapshots = []

def capture_callback(pipe, step, timestep, callback_kwargs):
    if step in SNAP_STEPS:
        z = callback_kwargs['latents']
        with torch.no_grad():
            img_t = pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample
        img_t = (img_t / 2 + 0.5).clamp(0, 1)
        img_np = img_t[0].permute(1, 2, 0).cpu().float().numpy()
        snapshots.append((step, Image.fromarray((img_np * 255).astype('uint8'))))
    return callback_kwargs

generator = torch.Generator(device=device).manual_seed(SEED)
result = pipe(
    prompt=PROMPT, negative_prompt=NEG_PROMPT,
    num_inference_steps=NUM_STEPS, generator=generator,
    callback_on_step_end=capture_callback,
    callback_on_step_end_tensor_inputs=['latents'],
)
final_image = result.images[0]
print(f'✅ Done. Captured {len(snapshots)} frames.')

In [ ]:
n_cols = len(snapshots) + 1
fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.4))
fig.patch.set_facecolor('#0E0B1A')

for ax, (step, img) in zip(axes, snapshots):
    ax.imshow(img); ax.set_title(f'Step {step+1}', color='#B7AEDB', fontsize=9); ax.axis('off')

axes[-1].imshow(final_image)
axes[-1].set_title('Final', color='#06B6D4', fontsize=9, fontweight='bold')
axes[-1].axis('off')

fig.suptitle(f'Denoising: "{PROMPT}"', color='white', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 2 — VAE Latent Space & Prompt Interpolation

The VAE compresses each 512×512 image into a **64×64×4 latent code**.  
Linearly mixing two latent codes and decoding each mixture shows the latent space is **semantically continuous** — the image transitions smoothly between the two concepts.

In [ ]:
PROMPT_A = 'a serene Japanese zen garden at dawn, watercolour style'
PROMPT_B = 'a cyberpunk city at night with neon lights, cinematic'
INTERP_STEPS = 6

def gen_image(prompt, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    return pipe(prompt=prompt, num_inference_steps=30, generator=g).images[0]

def encode(img):
    t = torch.tensor(
        np.array(img.resize((512, 512))).astype('float32') / 127.5 - 1.0
    ).permute(2, 0, 1).unsqueeze(0).to(device).to(dtype)
    with torch.no_grad():
        return pipe.vae.encode(t).latent_dist.sample() * pipe.vae.config.scaling_factor

def decode(z):
    with torch.no_grad():
        t = pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample
    t = (t / 2 + 0.5).clamp(0, 1)
    return Image.fromarray((t[0].permute(1,2,0).cpu().float().numpy() * 255).astype('uint8'))

print('Generating anchor A …'); img_a = gen_image(PROMPT_A, seed=1)
print('Generating anchor B …'); img_b = gen_image(PROMPT_B, seed=2)
z_a, z_b = encode(img_a), encode(img_b)
print(f'✅ Latent shape: {z_a.shape}')

In [ ]:
# ── Interpolation strip ───────────────────────────────────────────────────────
alphas = np.linspace(0.0, 1.0, INTERP_STEPS)
interp = [(a, decode((1-a)*z_a + a*z_b)) for a in alphas]

fig, axes = plt.subplots(1, INTERP_STEPS, figsize=(3.2*INTERP_STEPS, 3.6))
fig.patch.set_facecolor('#0E0B1A')
for ax, (alpha, img) in zip(axes, interp):
    ax.imshow(img); ax.set_title(f'α={alpha:.2f}', color='#B7AEDB', fontsize=9); ax.axis('off')
fig.suptitle(f'"{PROMPT_A[:38]}…"  →  "{PROMPT_B[:38]}…"', color='white', fontsize=9, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Raw latent channels (what the VAE actually encodes) ───────────────────────
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.patch.set_facecolor('#0E0B1A')
titles = ['C0 (structure)', 'C1 (edges)', 'C2 (texture)', 'C3 (colour)']
for col, title in enumerate(titles):
    for row, (z, label) in enumerate([(z_a, 'Prompt A'), (z_b, 'Prompt B')]):
        axes[row, col].imshow(z[0, col].cpu().float().numpy(), cmap='RdYlBu', vmin=-3, vmax=3)
        axes[row, col].set_title(f'{label} — {title}', color='#B7AEDB', fontsize=8)
        axes[row, col].axis('off')
fig.suptitle('VAE Latent Channels (64×64×4)', color='white', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 3 — DALL·E via OpenAI API

Same subject, different style directives — shows how style tokens shift the output  
while semantic content stays fixed.

> **Setup:** in the Colab left sidebar click **🔑 Secrets**, add `OPENAI_API_KEY`, and enable notebook access.

In [ ]:
from openai import OpenAI

# Load key from Colab Secrets (falls back to environment variable)
if IN_COLAB:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

client = OpenAI()

DALLE_PROMPTS = [
    'A majestic lion wearing a crown of stars, oil painting style',
    'A majestic lion wearing a crown of stars, low-poly 3D render',
]

def gen_dalle(prompt):
    r = client.images.generate(model='gpt-image-1', prompt=prompt, size='1024x1024')
    return Image.open(BytesIO(base64.b64decode(r.data[0].b64_json)))

dalle_images = []
for p in DALLE_PROMPTS:
    print(f'Generating: "{p}"')
    dalle_images.append(gen_dalle(p))
print('✅ Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('#0E0B1A')
for ax, img, prompt in zip(axes, dalle_images, DALLE_PROMPTS):
    ax.imshow(img); ax.set_title(f'"{prompt}"', color='#B7AEDB', fontsize=9, wrap=True); ax.axis('off')
fig.suptitle('DALL·E (gpt-image-1) — Same subject, different style', color='white', fontsize=12)
plt.tight_layout(); plt.show()

---
## Section 4 — Semantic Prompt Similarity

**Sentence-Transformers** embeds each prompt into a 384-d vector.  
**Cosine similarity** between vectors tells us how semantically close two prompts are —  
and therefore how similar the generated images are likely to look.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

PROMPTS = [
    'a serene mountain lake at sunrise',
    'rolling green hills under a cloudy sky',
    'a dense tropical rainforest with waterfalls',
    'a futuristic city skyline at night with neon lights',
    'a crowded Tokyo street in the rain',
    'a cyberpunk alleyway with holographic signs',
    'a portrait of a medieval knight in full armour',
    'close-up portrait of a samurai warrior',
    'a Victorian lady in a lavender dress',
    'abstract geometric shapes in vibrant colours',
    'swirling fractal patterns in blue and gold',
]

model_st  = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_st.encode(PROMPTS, normalize_embeddings=True)
sim_matrix = cosine_similarity(embeddings)
print(f'✅ Embedded {len(PROMPTS)} prompts  (dim={embeddings.shape[1]})')

In [ ]:
short = [p[:32]+'…' if len(p)>32 else p for p in PROMPTS]

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#0E0B1A'); ax.set_facecolor('#0E0B1A')
sns.heatmap(sim_matrix, xticklabels=short, yticklabels=short,
            cmap='magma', vmin=0, vmax=1, annot=True, fmt='.2f',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'cosine similarity'})
ax.set_title('Prompt Semantic Similarity', color='white', fontsize=13, pad=14)
plt.xticks(rotation=45, ha='right', color='#B7AEDB', fontsize=8)
plt.yticks(rotation=0, color='#B7AEDB', fontsize=8)
plt.tight_layout(); plt.show()
print('Bright cells = semantically similar prompts → likely similar outputs.')

In [ ]:
# ── Nearest-neighbour lookup — change QUERY to any prompt ────────────────────
QUERY = 'a snowy mountain peak at dawn'

q_emb  = model_st.encode([QUERY], normalize_embeddings=True)
scores = cosine_similarity(q_emb, embeddings)[0]
print(f'Query: "{QUERY}"\n')
for score, prompt in sorted(zip(scores, PROMPTS), reverse=True):
    print(f'  {score:.3f}  {"█"*int(score*20):<20}  {prompt}')

---
## 🎯 Summary

| Section | Core concept |
|---------|-------------|
| 1 — Denoising | Generation = iterative noise removal guided by a text prompt |
| 2 — Latent Space | Images live in a compact continuous space; blending latents blends images |
| 3 — DALL·E API | Same diffusion power, abstracted to a single API call |
| 4 — Prompt Similarity | Semantically close prompts → visually similar outputs |